# Imports

In [ ]:
# my functions
import helper_functions as hf

# data handling 
import pandas as pd 
import geopandas as gpd
import numpy as np

# plotting 
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns 

# mesa stuff
import mesa
from mesa import Agent, Model
from mesa.space import ContinuousSpace
from mesa.visualization import SolaraViz, make_plot_component, make_space_component
from mesa.visualization.utils import update_counter
from mesa.datacollection import DataCollector
from mesa.batchrunner import batch_run
#import solara

# rando stuff
import time
from scipy.stats import truncnorm
from scipy.stats import skewnorm
import random

# set notebook options
pd.set_option('display.max_rows', 1000)


# Load Road 

In [ ]:
#road_gdf = gpd.read_file("data/roads/hw210_w_speed_limits.geojson")
road_gdf = gpd.read_parquet("data/roads/hw210_sl_and_curvs.parquet")
display(road_gdf.head(3))


hf.make_colored_road_plot(road_gdf, 'curvature')


# Adjust Speed Helper Functions

In [ ]:

 def build_empirical_accel_function(pctile, mean_shift=-.2 , var_streach=1.3):
    """
    Builds a function that estimates acceleration (in m/s²)
    given speed (in mph), using empirical acceleration data
    from real-world stop sign behavior.

    Returns:
        accel(speed_mph): callable function
    """
    trimmed_pctile = np.clip(pctile, .07, .95)
    og_means = [1, 2.5, 2, 1.5]
    og_vars = [.35, .4, .4, .3]

    means = [i+mean_shift for i in og_means]
    var = [i*var_streach for i in og_vars]
    
    # differnet dists in m/s^s 
    dist0 = skewnorm(loc=means[0], scale=var[0], a=1)
    dist1 = skewnorm(loc=means[1], scale=var[1], a=1)
    dist2 = skewnorm(loc=means[2], scale=var[2], a=1)
    dist3 = skewnorm(loc=means[3], scale=var[3], a=1)

    # Acceleration values in G, time intervals in seconds
    segments = [
        {"start_t": 0, "end_t": 2, "accel_mpss": dist0.ppf(trimmed_pctile)},
        {"start_t": 2, "end_t": 4, "accel_mpss": dist1.ppf(trimmed_pctile)},
        {"start_t": 4, "end_t": 6, "accel_mpss": dist2.ppf(trimmed_pctile)},
        {"start_t": 6, "end_t": 8, "accel_mpss": dist3.ppf(trimmed_pctile)},
    ]

    # Convert to speed ranges in mph
    speed_bounds = [0]
    for seg in segments:
        delta_v_mps = seg["accel_mpss"] * (seg["end_t"] - seg["start_t"])
        delta_v_mph = delta_v_mps * 2.2  # convert m/s to mph
        speed_bounds.append(speed_bounds[-1] + delta_v_mph)

    # Pre-compute acceleration in m/s² for each segment
    accels_mps2 = [seg["accel_mpss"] for seg in segments]
    def accel(speed_mph):
        for i in range(len(speed_bounds) - 1):
            if speed_bounds[i] <= speed_mph < speed_bounds[i + 1]:
                return accels_mps2[i]
        return np.clip(accels_mps2[-1]*  (1-((speed_mph-speed_bounds[-1])/70)), 0,10)

    return accel


def vert_decel(slope_deg):
    slope_rad = np.radians(slope_deg)
    vert_decel = 9.81* np.sin(slope_rad)
    return vert_decel

# RoadSegmentAgent

In [ ]:
class RoadSegmentAgent(mesa.Agent):
    """Represents a segment of the road. Only one car can occupy it at a time."""
    
    def __init__(self, model, position, speed_limit, curvature, linked_coord):
        super().__init__(model)
        self.position = position  # The index of the segment
        self.occupied = False  # Whether a car is on this segment
        self.status = 'im just a road'
        self.speed_limit = speed_limit
        self.curvature = curvature
        self.linked_coord = linked_coord

    def adjust_speed(self):
        """Tracks occupancy but does not move."""
        pass
        
    def move_along_path(self):
        pass

# VehicleAgent base class

In [ ]:
class VehicleAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.status = "driving"
        self.speed = hf.get_mps(1) # starting speed
        self.break_cooldown = 0
        
        # For data collection 
        self.speed_change = 0
        self.created_at_step = self.model.steps 
        self.steps_taken = 0 
        self.distance_traveled = 0
        self.car_interactions = 0
        self.gap = 0 
        self.next_agent = None
        self.driving_action = None

        # Speed control tuning parameters (can be overridden)
        self.ideal_distance_multiplier = None
        self.acceptable_over = None
        self.curve_responce = None
        self.performance = None
        self.accel_curve = None
        


        # init all the road segement data 
        self.road_segments = self.model.agents.select(agent_type=RoadSegmentAgent)

        # Establish the Vehicles positio n 
        self.path = self.road_segments.get('position')
        self.path_index = 0
        self.model.space.place_agent(self, self.path[0])  # <-- here is the intial place agent

        
    def end_of_road(self):
        '''If at the last segment, remove the vehicle'''
        if self.path_index >= len(self.path) - 1:
            self.status = "arrived"
            
            self.model.finished_agents.append({
                "AgentID": self.unique_id,
                'AgentType': self.__class__.__name__,
                "created_at_step": self.created_at_step,
                "steps_taken": self.steps_taken,
                "car_interactions": self.car_interactions, 
                "distance_traveled": self.distance_traveled, 
                "approx_average_mph": hf.meters_to_miles(self.distance_traveled)/(self.steps_taken/3600), 
                'performance': self.performance,
                'curve_responce': self.curve_responce,
                "acceptable_over": hf.get_mph(self.acceptable_over),
                "ideal_distance_multiplier":self.ideal_distance_multiplier
            # Add more if needed
            })
            self.remove() 
            return True


    def get_next_agent(self): 
        '''
        Used in the get_gap function
        Takes self, checks if a next_agent exists and status == driving, if so uses that, if not trys to find a new next agent. 
        '''
        # check if 1) next car is already saved & 2)it is driving. This works because self.next_agent existing is tested first
        if self.next_agent and self.next_agent.status == "driving":
             return
        
        # if the next agent does not exist look for a new next_agent
        other_vehicles = self.model.agents.select(agent_type=VehicleAgent)
        cars_ahead = [
            agent for agent in other_vehicles
            if agent.distance_traveled > self.distance_traveled
        ]
        
        # set the next agent to the next vehicle, if no next vehicle then set to None
        if cars_ahead:
            self.next_agent = min(cars_ahead, key=lambda agent: agent.distance_traveled)
        else:
            self.next_agent = None
            
        
    def get_gap(self):
            """
            Returns:
                ideal_gap: float — the desired following distance (deg)
                gap: float — distance to the closest vehicle ahead (deg)

            Used in the adjust_speed function
            """
            ideal_gap = max(self.speed * self.ideal_distance_multiplier,2)
            
            # run the get_new_next_agent function 
            self.get_next_agent()
            
            # if a agent exists then measure the gap
            if self.next_agent: 
                gap = model.space.get_distance(self.pos, self.next_agent.pos)
            else:
                gap = np.nan
            self.gap = gap
            return ideal_gap, gap
        
    def get_speed_limit(self):
        def curve_adjust(max_affect_pct=0.5, curvature=45, speed=60):
            '''
            max_affect_pct - float: max possible speed reduction proportion at extreme curve
            curvature - float: standardized curve (0-90 degrees ideally)
            speed - float: current vehicle speed in mph
            '''
            curve_effect = curvature/90  # normalized curvature
            curve_effect = np.clip(curve_effect, 0, 1)  # protect against overcurve
        
            if speed <= 15:
                speed_effect = 0  # no curve penalty below 15 mph
            else:
                speed_effect = (speed - 10) / (60 - 10)  # normalized to [0, 1] between 15 and 60 mph
                speed_effect = np.clip(speed_effect, 0, 1)  # protect against overspeed
            return speed * (1 - (max_affect_pct * curve_effect * speed_effect))

        # gather the data from the road segments
        # 1. Get the 5 agents
        next_road_agents = list(self.road_segments)[self.path_index:self.path_index+4]
        posted_limit = [agent.speed_limit for agent in next_road_agents]
        curvatures = [agent.curvature for agent in next_road_agents]
        
        # weighted averages
        weights = np.array([1 / (1 + i) for i in range(len(posted_limit))])
        average_posted_limit = np.average(posted_limit, weights=weights)
        average_curvature = np.average(curvatures, weights=weights)

        # enter the info in to the curve adjust function 
        implicit_speed_limit = curve_adjust(self.curve_responce, average_curvature, average_posted_limit)
        
        return  hf.get_mps(implicit_speed_limit + self.acceptable_over)

    def less_smooth_brake(self, gap, ideal_gap):
        """
        Simulate more realistic, human-like braking behavior.
        Returns a value between 0 and 1 indicating brake intensity.
        """
        if ideal_gap <= 0 or np.isnan(ideal_gap):
            return 0
        force = max((ideal_gap - gap) / ideal_gap, 0)
        # Squared to overreact when too close
        base = force ** 2
        
        # Add some human-like noise
        noise = np.random.normal(0, .1)
        break_pct =  np.clip(base + noise, 0, 1)


        deceleration = break_pct * 8 # <- this is acting as max decel in mps 
        return deceleration

    def speed_limit_brake(self, speed_limit, speed):
        '''
        used when the car is going over the speed limit, essentially the more your going over the sl the more you apply breaks
        '''
        if speed < speed_limit:
            # this should never be triggered but i added anyway to make sure it didnt trip an error
            return 0
        mph_over = hf.get_mph(speed)-hf.get_mph(speed_limit) # used pct over because speed_limit and speed come in in mps 
        #print(mph_over)
        if mph_over > 7: 
            return 1.1
        elif mph_over > 2:
            return .5
        elif mph_over > 0:
            return .2
        # ~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~- The initial adjust speed ~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-~-
            
    def adjust_speed(self):
        ''' takes self from self uses'''
        ideal_gap, gap  = self.get_gap() 
        speed_limit = self.get_speed_limit()

        # save the current speed 
        old_speed = self.speed 
        
        # 1) measues the gap to the next vehicle, if less than the ideal gap, applies the smooth breaking
        if gap < ideal_gap:
            self.driving_action = 'smooth_break'
            self.car_interactions += 1
            self.break_cooldown = 5
            self.speed -= self.less_smooth_brake(gap=gap, ideal_gap=ideal_gap)

        # 3) if outside the jitter threashhold see if the car is above speed limit, if so break
        elif self.speed > speed_limit:
            self.driving_action = 'speed_limit_break'
            self.break_cooldown = 3
            self.speed -= self.speed_limit_brake(speed_limit=speed_limit, speed=self.speed)
            
        # 4) if outside the jitter threashhold & below speed limit & max speed then speed up 
        elif self.break_cooldown in [4,5]:
            self.driving_action = 'coast'
            self.break_cooldown -= 1 
        
        elif self.break_cooldown in [1,2,3]: # self.break_cooldown will be 3,2,1
            self.driving_action = 'slow_accelerate'
            self.speed+= self.accel_curve(hf.get_mph(self.speed)) * ((4-self.break_cooldown)/4)
            self.break_cooldown -= 1 

        else:
            self.driving_action = 'accelerate'
            self.speed+= self.accel_curve(hf.get_mph(self.speed))

        # overwrites
        if (self.next_agent is not None) and ((self.speed - self.next_agent.speed) > gap):
            self.driving_action = 'prevent_pass'
            self.break_cooldown = 5
            self.speed = self.next_agent.speed-1

        # dont go backwards
        self.speed = max(self.speed, 0)
        
        # new speed - old speed
        self.speed_change = self.speed - old_speed
    
            
    def move_along_path(self):
        """Move the agent along its predefined path based on current speed."""
        distance_to_travel = self.speed # sets a local variable in the function 
        self.distance_traveled += distance_to_travel # adds distance_to_travel(from this step) to the overall distance traveled
        pos = np.array(self.pos) # current position 
        new_position = pos
        
        while distance_to_travel > 0 and not self.end_of_road():
            next_target = np.array(self.path[self.path_index + 1])
            
            direction = model.space.get_heading(pos, next_target)
            distance = model.space.get_distance(pos, next_target)
    
            if distance < distance_to_travel:
                self.path_index += 1
                distance_to_travel -= distance
                pos = next_target
                new_position = pos
            else:
                step_vector = distance_to_travel * direction / distance
                new_position = pos + step_vector
                distance_to_travel = 0

        self.steps_taken += 1  
        self.model.space.move_agent(self, tuple(new_position))




# Specific VehicleAgent Classes

In [ ]:
class CarAgent(VehicleAgent):
    """Represents a car moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.acceptable_over =  hf.make_truncnorm(15, -2, 4, mean=3).rvs()  # this is a right skewed normal dist bounded by (-2,20)
        self.ideal_distance_multiplier = hf.make_truncnorm(2, .6, .3, mean=1).rvs()
        self.performance = np.random.uniform()
        self.accel_curve = build_empirical_accel_function(self.performance)
        self.curve_responce = hf.make_truncnorm(.95, .6, .1, mean=None).rvs() 
        
        
class BusAgent(VehicleAgent):
    """Represents a bus moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.acceptable_over = 0
        self.ideal_distance_multiplier = 2
        self.performance = .1
        self.accel_curve = build_empirical_accel_function(self.performance)
        self.curve_responce = .9 


        
        

    

In [ ]:
#sns.histplot(np.random.uniform())



# Model 

In [ ]:

class TrafficModel(mesa.Model):
    """Mesa model simulating traffic on the canyon road with a car cap."""

    def __init__(self, road_points_gdf=None, max_steps=50000, seed=123, log_agents=False, p_generate=.001, max_persons=50, bus_interval=30, car_preference=1):
        super().__init__(seed=seed)
        #model perams
        self.log_agents = log_agents
        self.road_points_gdf = road_points_gdf
        self.start_point = self.road_points_gdf.iloc[0].geometry.coords[0]  
        self.max_steps = max_steps
    
        # car perams
        self.p_generate = p_generate  # Probability of new car each step
        self.max_persons = max_persons  # Maximum number of persons allowed
        
        
        # bus perams
        self.bus_interval = bus_interval
        if self.bus_interval == 0: 
            self.car_preference = 1 
        else: 
            self.car_preference = car_preference
        
        self.bus_capacity = 30
        self.at_bus_stop = 0 
        self.bus_first_departure = self.random.randint(0, 5 * 60)  # Random step between 0 and 5 mins
        self.bus_generation_started = False
        
        
        # verious trackers
        self.too_close_counter = 0 
        self.person_counter = 0 
        self.bus_counter = 0 
        self.car_counter = 0 
        self.finished_agents = [] 
            
        # Set up ContinuousSpace
        buffer = .0001
        minx, miny, maxx, maxy = road_points_gdf.total_bounds
        self.space = ContinuousSpace(
            x_min=minx - buffer,
            x_max=maxx + buffer,
            y_min=miny - buffer,
            y_max=maxy + buffer,
            torus=False
        )

        # Create road segment agents - this just creates them in a loop setting the position via the gdf point
        self.road_segments = RoadSegmentAgent.create_agents( 
            model=self, 
            n=len(self.road_points_gdf), 
            position=[(point.x, point.y) for point in self.road_points_gdf.geometry], # need to be passed as a list
            speed_limit=[speed_limit for speed_limit in self.road_points_gdf.speed_limit],
            curvature = [curvature for curvature in self.road_points_gdf.curvature],
            linked_coord=[linked_coord for linked_coord in self.road_points_gdf.linked_coord]
        )
        # place all the road segments in space - goes hand in hand with read point reation 
        for agent, point in zip(self.road_segments, road_points_gdf.geometry):self.space.place_agent(agent, (point.x, point.y))

        # establish the data collector 
        agent_reporters={
            "AgentType": lambda a: a.__class__.__name__ ,
            #'status': lambda a: a.status if isinstance(a, VehicleAgent) else None,
            'distance_traveled': lambda a: a.distance_traveled if hasattr(a, 'distance_traveled') else None,
            'driving_action': lambda a: a.driving_action if isinstance(a, VehicleAgent) else None,
            'speed_change': lambda a: hf.get_mph(a.speed_change) if isinstance(a, VehicleAgent) else None,
            'speed_change_mps2': lambda a: a.speed_change if isinstance(a, VehicleAgent) else None,
            'speed': lambda a: hf.get_mph(a.speed) if isinstance(a, VehicleAgent) else None,
            'speed_mps': lambda a: a.speed if isinstance(a, VehicleAgent) else None,
            'steps_taken': lambda a: a.steps_taken if isinstance(a, VehicleAgent) else None,
            'gap_m':lambda a: a.gap if isinstance(a, VehicleAgent) else None,
            'ideal_gap_m':lambda a: a.speed * a.ideal_distance_multiplier if isinstance(a, VehicleAgent) else None,
            "next_vehicle": lambda a: a.next_agent.unique_id if isinstance(a, VehicleAgent) and a.next_agent is not None else None,
            'pos':lambda a: a.pos if isinstance(a, VehicleAgent) else None # dont remove, need for visuals
        }

        model_reporters={
            "max_persons": lambda m: m.max_persons,
            "p_generate": lambda m: m.p_generate, 
            "too_close_counter": lambda m: m.too_close_counter, 
            "FinishedAgentsSummary": lambda m: None  # Placeholder
        }

        if log_agents:
            self.datacollector = DataCollector(
                model_reporters = model_reporters, 
                agent_reporters = agent_reporters
            )
        else: 
            self.datacollector = DataCollector(model_reporters = model_reporters)
    
    # def generate_new_car(self):
    #     # Only generate if under max limit
    #     if self.car_counter >= self.max_cars:
    #         return
    #     # Get the starting point
    #     start_point = self.road_points_gdf.iloc[0].geometry.coords[0]  
    #     # Check if another car is too close to the start
    #     too_close = any(
    #         self.space.get_distance(agent.pos, start_point) < 1 # 
    #         for agent in self.agents.select(agent_type=VehicleAgent)[-5:] # last 5 cars
    #     )
    #     if too_close:
    #         self.too_close_tracker += 1
    #     elif self.random.random() < self.p_generate:
    #         CarAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
    #         self.car_counter += 1
    
    def generate_new_bus(self):
        """
        Generate a new bus:
        - First bus is generated at a random step (0–15 mins).
        - Then follow a fixed interval based on bus_interval (in minutes).
        - Never exceed max_buses on the road.
        """
        if self.person_counter >= self.max_persons:
            return

        if self.bus_interval == 0:
            return 
        current_step = self.steps
        steps_per_interval = self.bus_interval * 60
    
        # First departure check
        if not self.bus_generation_started:
            if current_step >= self.bus_first_departure:
                self.bus_generation_started = True
            else:
                return  # Still waiting for the randomized first departure
    
        # After the first departure
        if (current_step - self.bus_first_departure) % steps_per_interval == 0:    
            BusAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
            self.bus_counter += 1
            self.person_counter += self.at_bus_stop
            self.at_bus_stop = 0 
                
    def generate_person(self):
        # Only generate if under max limit
        if self.person_counter >= self.max_persons:
            return

        if self.person_counter > 0 and self.space.get_distance(self.agents.select(agent_type=VehicleAgent)[-1].pos, self.start_point) < 1:
            self.too_close_counter += 1
            return 

        if self.random.random() < self.p_generate:
            if (self.random.random() < self.car_preference) or (self.at_bus_stop >= self.bus_capacity):
                CarAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
                self.person_counter += 1
            else:
                self.at_bus_stop+=1 
                    
                
          

        
    def model_stop_process(self):
        # add agent summary data to the datacollector
        self.datacollector.model_vars["FinishedAgentsSummary"][-1] = self.finished_agents
        self.running = False
        
    def step(self):
        # generate bus 
        self.generate_person()
        # generate a new car based on a simple probability 
        self.generate_new_bus()
        
        # Shuffle agent execution and step them - this calls the step functions of the agents
        self.agents.do("adjust_speed")
        self.agents.do("move_along_path")
         
        # Collect data before stepping
        self.datacollector.collect(self)       

        # Stop model when all generated cars have been removed
        if self.person_counter == self.max_persons:
            remaining_vehicles = self.agents.select(agent_type=VehicleAgent)
            if len(remaining_vehicles) == 0:
                print(f"{self.person_counter} people generated stopping model.")
                self.model_stop_process()
        
        # Stop model at hard cap of steps
        if self.steps >= self.max_steps:
            print(f"Reached max step count ({self.max_steps}). Stopping model.")
            self.model_stop_process()
             


# Simple model run (fast)

In [ ]:
%%time
model = TrafficModel(road_points_gdf=road_gdf, max_steps=30000, log_agents=True, seed=123, p_generate=0.1, max_persons=100, bus_interval=15, car_preference=.5)

while model.running: 
    model.step()

print(f'Model ran for {model.steps} steps')

In [ ]:
model.car_preference

## Analyze data 
### Finished agents

In [ ]:
# process the finished_agents data 
finished_agents = hf.make_finished_agents_df(model)
vehicles_full = hf.make_vehicles_full_df(model)

### Agent level analysis

In [ ]:
# run the animation
# looking at one car
issue_car_id =  None

hf.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=10, watch=issue_car_id, zoom=50)

In [ ]:
hf.animate_relative_distance(vehicle_df=vehicles_full, agent_id=issue_car_id, distance_behind=100)

In [ ]:
#display(vehicles_full.head(3))
#sns.scatterplot(data=vehicles_full, x='speed', y='ideal_gap_m')
#sns.scatterplot(data=vehicles_full.loc[vehicles_full.gap_m <300], x='ideal_gap_m', y='gap_m')
#sns.scatterplot(data=vehicles_full, x='distance_traveled', y='speed')

In [ ]:
issue_step = 3000

### Issue car analysis

In [ ]:
issue_car_df = vehicles_full.loc[(vehicles_full.AgentID==issue_car_id+1)]
display(sns.scatterplot(data=issue_car_df, x='Step', y='speed', hue='driving_action', alpha=0.5))

print(f'Start of issue: {issue_car_df.loc[issue_car_df.speed<10]["Step"].min()}')

#issue_car_df.loc[issue_car_df.Step > issue_step].head(50)

In [ ]:
# looking at a group of issue cars
issue_car_ids = [issue_car_id, issue_car_id+1, issue_car_id+2]
issue_car_df = vehicles_full.loc[vehicles_full.AgentID.isin(issue_car_ids)]
print(issue_car_ids)
issue_car_df.loc[issue_car_df.Step > issue_step].head(10)

In [ ]:
def plot_agent_trajectories(df, y_var,step_range=(None, None)):
    """
    Plot agent trajectories over time using seaborn lineplot.

    Parameters:
    - df: pd.DataFrame with columns ['Step', 'AgentID', y_var]
    - y_var: str, the column to use on the y-axis

    Returns:
    - Displays a line plot where each line is one AgentID

    """
    start, end = step_range

    if start is not None:
        df = df[df['Step'] >= start]
    if end is not None:
        df = df[df['Step'] <= end]
    
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df, x="Step", y=y_var, hue="AgentID", legend=False)

     # Add labels to the start of each line
    for agent_id in df['AgentID'].unique():
        agent_df = df[df['AgentID'] == agent_id]
        first_point = agent_df.iloc[0]
        label = f"Agent:{agent_id} - ({first_point['AgentType']})"
        plt.text(first_point["Step"], first_point[y_var]+.5, label, fontsize=8, ha='left', va='top')

    plt.title(f"Agent Trajectories of {y_var} over Time", fontsize=14)
    plt.xlabel("Step")
    plt.ylabel(y_var)
    plt.tight_layout()
    plt.show()

step_range=(0,500
           )
plot_agent_trajectories(issue_car_df, 'speed')

In [ ]:
vehicles_full.head()

In [ ]:
# car gap over time

# Filter out rows where 'gap_ft' is not finite
vehicles_clean = vehicles_full[np.isfinite(vehicles_full['gap_m'])]
# Then aggregate
gap_over_time = vehicles_clean.groupby('Step', as_index=False)['gap_m'].mean()

sns.scatterplot(data=gap_over_time, x='Step', y='gap_m')
plt.xlabel("Step")
plt.ylabel('Mean Gap(m) by Step ')
plt.ylim(0,2000) 
#plt.xlim(0,500) 


gap_over_time